In [0]:
import os, glob, json, shutil
 
V1 = "/Volumes/ebi-ai-project/bills/landing"
RAW = "/Volumes/ebi-ai-project/bills_v2/raw"
for folder in ["truth", "_stash/wave2", "_stash/vendor", "lab/al_src", "_al"]:
    os.makedirs(f"{RAW}/{folder}", exist_ok=True)
 
def stage(templates, dest):
    """Copy the answer keys of these templates to dest, as T3__Invoice0001.json."""
    n = 0
    for t in templates:
        for src in sorted(glob.glob(f"{V1}/{t}/*.json")):
            shutil.copy(src, f"{RAW}/{dest}/{t}__{os.path.basename(src)}")
            n += 1
    return n
 
def read_text(path):
    for enc in ("utf-8-sig", "latin-1"):
        try:
            with open(path, encoding=enc) as f:
                return f.read()
        except UnicodeDecodeError:
            continue
 
def make_vendor_files(key, value, n, dest, suffix):
    """Copy n T6 answer keys plus one new top-level field: the vendor changed format."""
    made = 0
    for src in sorted(glob.glob(f"{V1}/T6/*.json"))[:n]:
        doc = json.loads(read_text(src))
        assert isinstance(doc, dict), "top level is not an object: tell me (B0.4)"
        doc[key] = value
        name = "T6__" + os.path.basename(src).replace(".json", f"_{suffix}.json")
        with open(f"{RAW}/{dest}/{name}", "w", encoding="utf-8") as f:
            json.dump(doc, f, ensure_ascii=False)
        made += 1
    return made
 
print("wave 1 -> truth/        :", stage(["T1", "T2", "T3"], "truth"))
print("wave 2 -> _stash/wave2/ :", stage(["T4", "T5", "T6"], "_stash/wave2"))
print("vendor -> _stash/vendor/:",
      make_vendor_files("Z1", "Tarifa verde", 5, "_stash/vendor", "z1"))
